# Model Evaluation: YOLOv8n + CBAM + SimAM Hybrid
**Experiment ID:** `cbam_simam`  
**Architecture:** CBAM + SimAM Hybrid  
**Dataset:** Helmet Detection (2 classes: helmet, non-helmet)  
> Run this notebook to evaluate. Point `WEIGHTS` to your `best.pt`.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
# ─────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────
EXPERIMENT_ID = "cbam_simam"
WEIGHTS       = r"C:\Users\soumy\OneDrive\Desktop\test(big)/../Testing_Scripts(final)/models/best (CBAM_SIMAM).pt"
DATA_YAML     = r"C:\Users\soumy\OneDrive\Desktop\test(big)/../Testing_Scripts(final)/data_big.yaml"
DATASET_ROOT  = r"C:\Users\soumy\OneDrive\Desktop\test(big)"
IMGSZ         = 640
BATCH         = 16
CONF_THRES    = 0.001
IOU_THRES     = 0.5
OUTPUT_DIR    = r"C:\Users\soumy\OneDrive\Desktop\test(big)/../Testing_Scripts(final)/eval_small_cbam_simam"
CLASS_NAMES   = ["helmet", "non-helmet"]
# ─────────────────────────────────────────────

In [ ]:
!pip install ultralytics -q

In [ ]:
# ── CBAM+SimAM must be registered before loading weights ──
import torch, torch.nn as nn, sys

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool_h = nn.AdaptiveAvgPool2d(1)
        self.pool_m = nn.AdaptiveMaxPool2d(1)
        r = max(channels // reduction, 1)
        self.fc = nn.Sequential(nn.Conv2d(channels, r, 1, bias=False), nn.ReLU(inplace=True), nn.Conv2d(r, channels, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.fc(self.pool_h(x)) + self.fc(self.pool_m(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.conv(torch.cat([torch.mean(x,1,True), torch.max(x,1,True)[0]], 1)))

class SimAM(nn.Module):
    """SimAM: Parameter-Free Attention (ICML 2021)"""
    def __init__(self, e_lambda=1e-4):
        super().__init__()
        self.e_lambda = e_lambda
        self.act = nn.Sigmoid()
    def forward(self, x):
        b, c, h, w = x.size()
        n = w * h - 1
        x_minus_mu_square = (x - x.mean(dim=[2, 3], keepdim=True)).pow(2)
        y = x_minus_mu_square / (4 * (x_minus_mu_square.sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)) + 0.5
        return x * self.act(y)

class CBAMSimAM(nn.Module):
    """CBAM + SimAM Hybrid."""
    def __init__(self, kernel_size=7, e_lambda=1e-4):
        super().__init__()
        self.kernel_size = kernel_size
        self.e_lambda = e_lambda
        self.reduction = 16
        self.ca = None; self.sa = None
        self.simam = SimAM(e_lambda)
    def forward(self, x):
        if self.ca is None:
            c = x.shape[1]
            self.ca = ChannelAttention(c, self.reduction).to(x.device)
            self.sa = SpatialAttention(self.kernel_size).to(x.device)
        x = self.ca(x)
        x = self.sa(x)
        x = self.simam(x)
        return x

import ultralytics.nn.tasks as tasks, ultralytics.nn.modules as modules
for cls in [CBAMSimAM, ChannelAttention, SpatialAttention, SimAM]:
    setattr(tasks, cls.__name__, cls)
    setattr(modules, cls.__name__, cls)
    for s in ("block","conv","head"):
        if hasattr(modules,s): setattr(getattr(modules,s), cls.__name__, cls)
    sys.modules['__main__'].__dict__[cls.__name__] = cls
torch.serialization.add_safe_globals([CBAMSimAM, ChannelAttention, SpatialAttention, SimAM])
print("[✓] CBAMSimAM registered — safe to load weights.")

In [ ]:
import os, json, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from ultralytics import YOLO

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[✓] Output directory: {OUTPUT_DIR}")
print(f"[✓] Loading weights : {WEIGHTS}")
model = YOLO(WEIGHTS)


## 1 · Validation on Test Split

In [ ]:
test_results = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    conf=CONF_THRES,
    iou=IOU_THRES,
    save_json=True,
    plots=True,
    project=OUTPUT_DIR,
    name="test",
    exist_ok=True,
    verbose=True,
)

print("\n[Test] Raw metrics object:", test_results)

## 2 · Extract & Display Metrics

In [ ]:
def extract_metrics(results, split_name):
    """Extract a flat dict of metrics from a YOLO val results object."""
    box = results.box
    metrics = {
        "experiment":   EXPERIMENT_ID,
        "split":        split_name,
        "mAP50":        round(float(box.map50),  4),
        "mAP50_95":     round(float(box.map),    4),
        "precision":    round(float(box.mp),     4),
        "recall":       round(float(box.mr),     4),
        "f1":           round(float(np.mean(box.f1)), 4),
    }
    for i, cls_name in enumerate(CLASS_NAMES):
        try:
            metrics[f"AP50_{cls_name}"]    = round(float(box.ap50[i]),  4)
            metrics[f"AP50_95_{cls_name}"] = round(float(box.ap[i]),    4)
            metrics[f"P_{cls_name}"]       = round(float(box.p[i]),     4)
            metrics[f"R_{cls_name}"]       = round(float(box.r[i]),     4)
            metrics[f"F1_{cls_name}"]      = round(float(box.f1[i]),    4)
        except Exception:
            pass
    return metrics

test_metrics = extract_metrics(test_results, "test")

print("=" * 55)
print(f"  EXPERIMENT : {EXPERIMENT_ID.upper()}")
print("=" * 55)
print(f"\n  [TEST]")
print(f"    mAP@50        : {test_metrics['mAP50']}")
print(f"    mAP@50-95     : {test_metrics['mAP50_95']}")
print(f"    Precision     : {test_metrics['precision']}")
print(f"    Recall        : {test_metrics['recall']}")
print(f"    F1            : {test_metrics['f1']}")
for cls in CLASS_NAMES:
    print(f"    AP50 [{cls:<12}]: {test_metrics.get(f'AP50_{cls}', 'N/A')}")
print("=" * 55)

# Fitness
fitness = round(float(test_results.fitness), 4)
print(f"  Fitness         : {fitness}")

# Speed
speed = test_results.speed
print(f"  Preprocess      : {speed['preprocess']:.2f} ms/img")
print(f"  Inference       : {speed['inference']:.2f} ms/img")
print(f"  Postprocess     : {speed['postprocess']:.2f} ms/img")


## 3 · Save Metrics to CSV & JSON

In [ ]:
# Add fitness and speed to metrics
test_metrics["fitness"] = fitness
test_metrics["inference_ms_img"] = round(speed["inference"], 2)
test_metrics["preprocess_ms_img"] = round(speed["preprocess"], 2)
test_metrics["postprocess_ms_img"] = round(speed["postprocess"], 2)

all_metrics = [test_metrics]
csv_path = f"{OUTPUT_DIR}/metrics_{EXPERIMENT_ID}.csv"
df = pd.DataFrame(all_metrics)
df.to_csv(csv_path, index=False)
print(f"[✓] CSV saved  → {csv_path}")

json_path = f"{OUTPUT_DIR}/metrics_{EXPERIMENT_ID}.json"
with open(json_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"[✓] JSON saved → {json_path}")

# Also save standardized test_results.csv
std_csv = f"{OUTPUT_DIR}/test_results.csv"
std_row = {"Model Name": EXPERIMENT_ID, "Precision": test_metrics["precision"],
           "Recall": test_metrics["recall"], "F1": test_metrics["f1"],
           "mAP50": test_metrics["mAP50"], "mAP50_95": test_metrics["mAP50_95"],
           "Fitness": fitness, "Inference_ms_img": round(speed["inference"], 2)}
pd.DataFrame([std_row]).to_csv(std_csv, index=False)
print(f"[✓] Standardized CSV → {std_csv}")
df

## 4 · Metrics Bar Chart

In [ ]:
metric_keys   = ["mAP50", "mAP50_95", "precision", "recall", "f1"]
metric_labels = ["mAP@50", "mAP@50-95", "Precision", "Recall", "F1"]
test_vals = [test_metrics[k] for k in metric_keys]
x     = np.arange(len(metric_labels))
width = 0.5
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(x, test_vals, width, color="#DD8452", alpha=0.88)
ax.set_ylim(0, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title(f"Test Metrics — {EXPERIMENT_ID.upper()}", fontsize=14, fontweight="bold")
ax.grid(axis="y", linestyle="--", alpha=0.5)
for bar in bars:
    h = bar.get_height()
    ax.annotate(f"{h:.3f}", xy=(bar.get_x()+bar.get_width()/2, h),
                xytext=(0,3), textcoords="offset points", ha="center", fontsize=9)
plt.tight_layout()
chart_path = f"{OUTPUT_DIR}/metrics_chart_{EXPERIMENT_ID}.png"
plt.savefig(chart_path, dpi=150)
plt.show()
print(f"[✓] Chart saved → {chart_path}")

## 5 · Per-Class AP Bar Chart

In [ ]:
ap50_vals = [test_metrics.get(f"AP50_{c}", 0) for c in CLASS_NAMES]
ap95_vals = [test_metrics.get(f"AP50_95_{c}", 0) for c in CLASS_NAMES]
x     = np.arange(len(CLASS_NAMES))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - width/2, ap50_vals, width, label="AP@50",    color="#55A868", alpha=0.88)
b2 = ax.bar(x + width/2, ap95_vals, width, label="AP@50-95", color="#C44E52", alpha=0.88)
ax.set_ylim(0, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=12)
ax.set_ylabel("AP Score", fontsize=12)
ax.set_title(f"Per-Class AP (Test) — {EXPERIMENT_ID.upper()}", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.5)
for bar in [*b1, *b2]:
    h = bar.get_height()
    ax.annotate(f"{h:.3f}", xy=(bar.get_x()+bar.get_width()/2, h),
                xytext=(0,3), textcoords="offset points", ha="center", fontsize=9)
plt.tight_layout()
pc_path = f"{OUTPUT_DIR}/per_class_ap_{EXPERIMENT_ID}.png"
plt.savefig(pc_path, dpi=150)
plt.show()
print(f"[✓] Per-class chart saved → {pc_path}")

## 6 · Model Info

In [ ]:
info = model.info(verbose=True)
print(f"\n[✓] Model info printed above.")
print(f"    Weights file: {WEIGHTS}")

## 7 · Final Summary

In [ ]:
print("\n" + "═" * 60)
print(f"  FINAL SUMMARY — {EXPERIMENT_ID.upper()}")
print("═" * 60)
print(f"  {'Metric':<20} {'Test':>10}")
print("  " + "-" * 32)
for key, label in zip(metric_keys, metric_labels):
    print(f"  {label:<20} {test_metrics[key]:>10.4f}")
print("  " + "-" * 32)
print(f"  {'Fitness':<20} {fitness:>10.4f}")
print(f"  {'Inference ms/img':<20} {speed['inference']:>10.2f}")
print("═" * 60)
print(f"\n  Outputs saved to: {OUTPUT_DIR}/")